In [ ]:
import pandas as pd
import numpy as np
import volatility_factor as vf
from volatility_factor import VarData, GarchSim, VarBacktest

## <center>Volatility Factor Project</center>

<p><b>Overview:</b> the Volatility Factor project is a Python package (<code>volatility_factor</code>),
a configuration file (<code>configs/var_config_volfactor.yml</code>) for user-provided settings, and input
data (<code>data/pricing_data_combined.csv</code>). Users change the configuration file and run
<code>notebooks/volatility_factor_main.ipynb</code> to generate output. Please refer to
<i>GARCH Simulation - Volatility Factor - MDA Departmental Procedures</i> for operating procedures and
<i>GARCH Simulation - Technical Documentation</i> for technical details.</p>

<p><b>Purpose:</b> this document describes and illustrates the package's functionality. It is not
procedural; see the procedures document for production use.</p>

<p><b>Components</b> (modules of <code>volatility_factor</code>):</p>

**1. data:** `PricingData` (loads prices from CSV, parquet, a DataFrame or a Delta table) and the `VarData` class (prices, returns and descriptions for an index or weighted portfolio).

**2. garch:** the `GarchSim` class, which fits GARCH(1,1) models and simulates n-day VaR. Each fitted model is kept as a `GarchFit` wrapping the full `arch` result.

**3. nonparametric:** the `NonParametricVar` class (circular block bootstrap).

**4. backtesting:** the `VarBacktest` class, the Kupiec / Christoffersen test (`log_likelihood_test`) and `run_backtests` for whole result sets.

**5. monitoring:** quarterly F-tests, exception counts and shortfalls.

**6. config / periods / pipeline / plotting:** configuration loading, sample and stress windows, the config-driven runs, and Plotly charts.

---
---
---

## <center>VarData</center>

<p><b>Description:</b> <code>VarData</code> computes measures and descriptive attributes of index prices
(via cached properties). Pricing data is loaded on first use, from (in order) a <code>data=</code>
argument, <code>vf.set_data(...)</code>, the <code>VOLFACTOR_DATA</code> environment variable, or
<code>data_settings.pricing_data</code> in the config. Properties fall into two categories:</p>

**1. price measures:** prices, price returns, log returns, etcetera.

**2. descriptors:** ticker description, pricing index description, and the volatility factor id (looked up in the config, or given via `VarData.from_id(...)`).

<br> Below we provide some examples to illustrate some features of this class on a stand-alone basis.

---

**Example:** Instantiate VaRData

In [ ]:
vdata = VarData(tickers = ["CB10"], weights= None)
print(type(vdata))

---

**Example**: ticker data type validation checks

In [ ]:
vdata1 = VarData(tickers = "CB10") #<-------------Accepts string
print(type(vdata1))
vdata2 = VarData(tickers = ["CB10"]) #<-----------Accepts list
print(type(vdata2))
vdata3 = VarData(tickers = ("CB10",)) #<----------Accepts tuple
print(type(vdata3))
vdata4 = VarData.from_id("agy_cmo") #<------------Or a configured volatility_factor_id
print(vdata4.tickers, vdata4.volatility_factor_id)
#vdata = VarData(tickers=set("CB10")) #<----------UNCOMMENT TO VIEW ERROR

---

**Example:** weights value validation checks

In [ ]:
vdata = VarData(tickers = "CB10") #<---single-index w/o weights; weights value default to None - assigns 1 to weights
print(vdata.weights)
vdata = VarData(tickers = ["CB10", "CB20"]) #<-multi-index w/o weights; assigns equal weights
print(vdata.weights)
vdata = VarData(tickers = "CB10", weights = 2) #<--single-index with invalid weight; weights value defaults to 1 
print(vdata.weights)
#vdata = VarData(tickers = ["CB10", "CB20"], weights = [1,2]) #<-------UNCOMMENT TO VIEW ERROR
#print(vdata.weights)

---

**Example:** Available Properties<br>
The following properties are available using dot notation

In [ ]:
print("Index Pricing Ticker\n")
print(f"{vdata.tickers}")
print(f"{type(vdata.tickers)}")
print("---------------------------------\n")

print("Index Weights\n")
print(f"{vdata.weights}")
print(f"{type(vdata.weights)}")
print("---------------------------------\n")

print("Single Index Indicator\n")
print(f"{vdata.single_index}")
print(f"{type(vdata.single_index)}")
print("---------------------------------\n")

print("Description of Pricing Index\n")
print(f"{vdata.index_description}")
print(f"{type(vdata.index_description)}")
print("---------------------------------\n")

print("Description of Ticker Pricing\n")
print(f"{vdata.ticker_description}")
print(f"{type(vdata.ticker_description)}")
print("---------------------------------\n")

print("MDA-assigned Volatility Factor ID\n")
print(f"{vdata.volatility_factor_id}")
print(f"{type(vdata.volatility_factor_id)}")
print("---------------------------------\n")

print("Index Prices\n")
print(f"{vdata.price_matrix.head(2)}")
print(f"{type(vdata.price_matrix)}")
print("---------------------------------\n")

print("One-Day Price Change\n")
print(f"{vdata.price_return_matrix.head(2)}")
print(f"{type(vdata.price_return_matrix)}")
print("---------------------------------\n")

print("Weighted One-Day Price Change\n")
print(f"{vdata.wtd_price_return.head(2)}")
print(f"{type(vdata.wtd_price_return)}")
print("---------------------------------\n")

print("Logarithm of Price\n")
print(f"{vdata.log_price_matrix.head(2)}")
print(f"{type(vdata.log_price_matrix)}")
print("---------------------------------\n")

print("One-Day Log Return\n")
print(f"{vdata.log_return_matrix.head(2)}")
print(f"{type(vdata.log_return_matrix)}")
print("---------------------------------\n")

print("Weighted One-Day Log Return\n")
print(f"{vdata.wtd_log_return.head(2)}")
print(f"{type(vdata.wtd_log_return)}")
print("---------------------------------\n")

---
---
---

## <center>GarchSim</center>

<p><b>Description:</b> the <b>GarchSim</b> class estimates the Volatility Factor in two steps:
<b>(1) estimation of the model</b> and <b>(2) simulation</b>.</p>

<p><b>Estimation:</b> we use the <a href="https://arch.readthedocs.io/en/latest/" target="_top">ARCH</a>
package to estimate one GARCH(1,1) model per index. Each fitted model is kept in
<code>gsim.fits[ticker]</code> as a <code>GarchFit</code>, which exposes every attribute of the
<code>arch</code> result directly (for example <code>fit.aic</code>, <code>fit.summary()</code>,
<code>fit.forecast(...)</code>) plus the scale used and each fitting attempt.</p>

<br> Below we provide some examples to illustrate some features of this class on a stand-alone basis.

---

**Example:** Instantiate GarchSim

In [ ]:
gsim = GarchSim(tickers = ["M0A0"], weights = None, sample_start = "2013-10-01", sample_end = "2022-09-30")

---

### <center>Estimation</center>

Once instantiated, the following properties are available. They are cached and lazily
evaluated: the models are fitted the first time any of them is accessed (or when
<code>gsim.fit()</code> is called), and refitted if the sample dates change.

In [ ]:
print("Training data (one-day log-return)\n")
print(gsim.training_data.head(2))
print(f"{type(gsim.training_data)}")
print("---------------------------------\n")

print("Training data scaled (training data * y_scale)\n")
print(gsim.training_data_scaled.head(2))
print(f"{type(gsim.training_data_scaled)}")
print("---------------------------------\n")

print("Scale: y-scale value used to scale training data\n")
print(gsim.yscale)
print(f"{type(gsim.yscale)}")
print("---------------------------------\n")

print("Convergence Code: meta-data from loglikelihood optimization result\n")
print(gsim.convergence_codes)
print(f"{type(gsim.convergence_codes)}")
print("---------------------------------\n")

print("Coefficients\n")
print(gsim.coefficients)
print(f"{type(gsim.coefficients)}")
print("---------------------------------\n")

print("Conditional Volatility\n")
print(gsim.conditional_vol.head(2))
print(f"{type(gsim.conditional_vol)}")
print("---------------------------------\n")

print("Residuals\n")
print(gsim.residuals.head(2))
print(f"{type(gsim.residuals)}")
print("---------------------------------\n")

print("Standardized Residuals\n")
print(gsim.std_residuals.head(2))
print(f"{type(gsim.std_residuals)}")
print("---------------------------------\n")

print("Standardized Returns\n")
print(gsim.std_returns.head(2))
print(f"{type(gsim.std_returns)}")

---

Each fitted <code>arch</code> result is also kept. <code>gsim.fits</code> is keyed by ticker;
<code>gsim.garch_fit</code> is the same results as a list. Below, <code>_fit</code> is the
<code>GarchFit</code> for M0A0.

In [ ]:
_fit = gsim.fits["M0A0"] #<---GarchFit, keyed by ticker (gsim.garch_fit[0] is the raw arch result)

print("Estimation output\n")
display(_fit.summary())
print("---------------\n")

print("Coefficients\n")
display(_fit.params)
print("---------------\n")

print("Scale and fitting attempts (yscale, convergence flag)\n")
display(_fit.yscale, _fit.attempts)
print("---------------\n")

print("Summary across constituents\n")
display(gsim.summary())

In [ ]:
gsim.plot_fit()

---

### <center>Simulation</center>

Once the model is specified and fit using the Arch package, a simulation processes generates forward-looking one-day price paths.  The one-day price paths are then transformed to n-day price change paths as setforth in the model documentation.  The purpose of this section is to outline some of the different types of simulations the model was designed to accomodate.  These simulations are invoked by passing one of the following arguments to the <i>shock_type</i> parameter of the <i>CustomNDayPriceSimulation</i> method.<br>

**1. normal:** this simulation process draws the exogenous shocks from a standard normal distribution for the entire duraction of the forecast (horizon 1 + horizon 2 + ndays - 1).

**2. custom_normal:** <u><b>this is the simulation process used to derive the long-term VaR - the volatility factor used for the annual review period</u></b>.  It is idential to the "Normal" simulation - it draws the exogenous shocks from a standard normal distribution for the entire duration of the forecast.  However, we impose stress to the shocks for periods 64 through 68 by by multiplying them by $\sqrt{2}$.  This process is described in detail in section 3.3.1 of the <i>GARCH Simulation - Technical Documentation</i>.

**3. bootstrap:** Unlike the simulation processes described above - both of which assume $e_t$ follow a standard norml distribution - this process draws from the fitted values of $e_t$ from a user-defined date range.  When <i>shock_type</i> is set to <i>bootstrap</i>, users must provide two additional arguments to the <i>bootstrap_start</i> and the <i>bootstrap_end</i> parameters.

**4. custom_bootstrap:** This simulation process combines the <i>normal</i> and the <i>bootstrap</i> simulation processes described above.  For the first <i>horizon1</i> periods, shocks are drawn from the fitted values of $e_t$ from a user-defined date range passed to the <i>bootstrap_start</i> and the <i>bootstrap_end</i> parameters, and for the remainder of the simulation, the the shocks are drawn from a standard normal distribution.

#### <center>Simulation - Starting Values</center>

All of the different simulation types described above can be executed without setting the starting values of $ϵ_0$ and $α_0$.  When the <i>start_vals</i> parameter is unchanged from it's default value of None, the last fitted values for ϵ and α are used in the simulation.  To specify the starting values, a two-element list must be passed to the <i>start_vals</i> parameter; the first element with a date and the second element with a quantile.  When a user provides values for the <i>start_vals</i> parameter, a one-day variance forecast is performed for each business day from the date provided through the end of the sample period.  The values of $ϵ_0$ and $α_0$ are set to the corresponding quantile of the series of one-day variance forecasts.  <u><b>For the long-term VaR - the volatility factor used for the annual review period - the values of $ϵ_0$ and $α_0$ are set to the 50th quantile of the one-day variance forecast over the last year in the sampele period.</u></b>.

---

**Start:** to begin, we'll instantiate an instance of GarchSim using the same values used above.

In [ ]:
v = GarchSim(tickers = ["M0A0"], weights = None, sample_start = "2013-10-01", sample_end = "2022-09-30") #instantiate GarchSim

---

**Next:** next, we'll assign some values to variables to use in the CustomNDayPriceSimulation method.

In [ ]:
horizon1, horizon2, ndays, nsims = [63, 252, 10, 10000] #required args for CustomNDayPriceSimulation
horizon2 = horizon2 + ndays - 1 #For 'horizon2' parameter.
starting_values = ["2021-10-01", 0.50] #for 'start_vals' parameter

---

**Last:** we'll demonstrate the simulations outlined above.

---

#### <center>normal (no starting values set)</center>

In [ ]:
var = v.custom_nday_price_simulation(horizon1 = horizon1, horizon2 = horizon2, ndays = ndays, nsims = nsims, 
                                shock_type = "normal")
display(var)

#### <center>normal (WITH starting values set)</center>

In [ ]:
var = v.custom_nday_price_simulation(horizon1 = horizon1, horizon2 = horizon2, ndays = ndays, nsims = nsims, 
                                shock_type = "normal", start_vals=starting_values)
display(var)

---

#### <center>custom_normal (no starting values set)</center>

In [ ]:
var = v.custom_nday_price_simulation(horizon1 = horizon1, horizon2 = horizon2, ndays = ndays, nsims = nsims, 
                                shock_type = "custom_normal")
display(var)

#### <center>custom_normal (WITH starting values set)</center>

In [ ]:
var = v.custom_nday_price_simulation(horizon1 = horizon1, horizon2 = horizon2, ndays = ndays, nsims = nsims, 
                                shock_type = "custom_normal", start_vals=starting_values)
display(var)

---

#### <center>bootstrap (no starting values set)</center>

In [ ]:
var = v.custom_nday_price_simulation(horizon1 = horizon1, horizon2 = horizon2, ndays = ndays, nsims = nsims, 
                                shock_type = "bootstrap", bootstrap_start = "2021-04-01", bootstrap_end = "2022-03-31")
display(var)

#### <center>bootstrap (WITH starting values set)</center>

In [ ]:
var = v.custom_nday_price_simulation(horizon1 = horizon1, horizon2 = horizon2, ndays = ndays, nsims = nsims, 
                                shock_type = "bootstrap", bootstrap_start = "2021-04-01", bootstrap_end = "2022-03-31",
                                start_vals=starting_values)
display(var)

---

#### <center>custom_bootstrap (no starting values set)</center>

In [ ]:
var = v.custom_nday_price_simulation(horizon1 = horizon1, horizon2 = horizon2, ndays = ndays, nsims = nsims, 
                                shock_type = "custom_bootstrap", bootstrap_start = "2022-07-01", bootstrap_end = "2022-09-30")
display(var)

#### <center>custom_bootstrap (WITH starting values set)</center>

In [ ]:
var = v.custom_nday_price_simulation(horizon1 = horizon1, horizon2 = horizon2, ndays = ndays, nsims = nsims, 
                                shock_type = "custom_bootstrap", bootstrap_start = "2022-07-01", bootstrap_end = "2022-09-30",
                                start_vals=starting_values)
display(var)

---

---
---
---

## <center>VarBacktest</center>

<p><b>Description:</b> the <b>VarBacktest</b> class calculates backtesting measures and draws
Plotly charts. Settings (window, VaR, liquidation period) can be changed with
<code>with_var(...)</code> / <code>with_window(...)</code>, which clear the cached results.</p>

---

**Example:** Price Plot & Price Change Plot <br>

In [ ]:
bt = VarBacktest(tickers = "M0A0", n_liquidation = 10) #instantiate bt
bt.price_plot().show() #<------------default title; Note: this is only method that works without any optional arguments
bt.price_change_plot(title_text="This is how you change the Title").show() #<-----user-defined title

---

**Example:** Backtest plot

In [ ]:
bt = VarBacktest(tickers = "M0A0", n_liquidation = 10, var = 3, start_date="2020-01-01", end_date="2022-09-30")
bt.backtest_plot()

---

**Example:** Available Properties<br>
The following properties are available using dot notation.

In [ ]:
print("VaR Percentage\n")
display(bt.var_pct) 
print("---------------\n")

print("N-Day Arithmetic Return\n")
display(bt.nday_price_return.head(2))
print("---------------\n")

print("N-Day Log Return\n")
display(bt.nday_log_return.head(2))
print("---------------\n")

print("Breaks\n")
display(bt.break_series.head(2))
print("---------------\n")

print("Shortfalls\n")
display(bt.shortfall_series.head(2))

In [ ]:
print("Kupiec / Christoffersen tests\n")
display(bt.independence_test())